# CEMC and Li Percolation Example for Li1.2Mn0.4Ti0.4O2


## Dependencies

Install the required packages before running:

```bash
pip install git+https://github.com/Liaojh123/SROS.git
pip install git+https://github.com/CederGroupHub/smol.git
pip install git+https://github.com/atomisticnet/dribble.git
pip install pymatgen numpy pandas matplotlib
```

`SROS` generates the random starting structures and runs the CEMC workflow, `smol` is required by the cluster-expansion Monte Carlo code, and `dribble` performs the Li percolation calculation.

In [1]:
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Compatibility for dribble versions that still reference deprecated NumPy aliases.
if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "product"):
    np.product = np.prod

try:
    from SROS import CEMC, generate_random_any_composition as random_structure
except ModuleNotFoundError:
    from sros.calculation import CEMC
    from sros.calculation import generate_random as random_structure
from smol.io import load_work
from smol.moca import Ensemble, Sampler
from dribble.io import Input
from dribble.lattice import Lattice
from dribble.percolator import Percolator

# Run this notebook from the examples/ directory. If you run it elsewhere,
# replace these paths with absolute paths, e.g. /home/ljh/.../model.mson.
ce_model_path = "../models/Li-Mn-Ti-O_clustre_expansion_model.mson"
output_dir = "cemc_percolation_Li1p2Mn0p4Ti0p4O2"

li_content = 1.2
mn_content = 0.4
temperatures = [100, 1000, 10000, 100000]
N_STRUCTURES = 1
MC_STEPS = 1000
SC_MATRIX = np.array([[5, 0, 0], [0, 6, 0], [0, 0, 8]])
BASE_SEED = 202502

Path(output_dir).mkdir(parents=True, exist_ok=True)
print(f"CE model: {ce_model_path}")
print(f"Output directory: {output_dir}")

CE model: ../models/Li-Mn-Ti-O_clustre_expansion_model.mson
Output directory: cemc_percolation_Li1p2Mn0p4Ti0p4O2


## Generate CEMC Structures

Each structure starts from a random Li-Mn-Ti-O DRX configuration and is relaxed by CEMC at the target temperature using the provided cluster-expansion model.

In [2]:
def make_initial_structure(seed):
    random.seed(seed)
    np.random.seed(seed)
    if hasattr(random_structure, "make_supercell"):
        supercell = random_structure.make_supercell(scaling_matrix=SC_MATRIX.tolist())
    else:
        supercell = random_structure.make_supercell_matrix(scaling_matrix=SC_MATRIX.tolist())
    return random_structure.modify_structure(li_content, mn_content, supercell)


def run_cemc_structure(initial_structure, temperature):
    work = load_work(ce_model_path)
    expansion = work["ClusterExpansion"]
    ensemble = Ensemble.from_cluster_expansion(expansion, SC_MATRIX)
    sampler = Sampler.from_ensemble(ensemble, temperature)

    mn2_ratio, mn3_ratio, mn4_ratio = CEMC.calculate_mn_ratios(li_content, mn_content)
    initial_occupancies = CEMC.assign_element_numbers(initial_structure, mn2_ratio, mn3_ratio, mn4_ratio)
    sampler.run(MC_STEPS, initial_occupancies=initial_occupancies, thin_by=1000, progress=False)

    occupancies = sampler.samples.get_occupancies()
    if getattr(occupancies, "ndim", 1) > 1:
        occupancies = occupancies[-1]
    structure = ensemble.processor.structure_from_occupancy(occupancies)

    if hasattr(CEMC, "reorder_atoms_flexible"):
        return CEMC.reorder_atoms_flexible(structure, ["Li", "Mn", "Ti", "O"])
    return structure


def generate_cemc_structures(overwrite=False):
    structure_files = []
    for temperature in temperatures:
        temp_dir = Path(output_dir) / f"{temperature}K"
        temp_dir.mkdir(parents=True, exist_ok=True)
        for i in range(N_STRUCTURES):
            out_file = temp_dir / f"CEMC_{i:03d}.vasp"
            if overwrite or not out_file.exists():
                seed = BASE_SEED + temperature + i
                initial_structure = make_initial_structure(seed)
                cemc_structure = run_cemc_structure(initial_structure, temperature)
                cemc_structure.to(str(out_file), fmt="poscar")
            structure_files.append((temperature, i, out_file))
    return structure_files


structure_files = generate_cemc_structures(overwrite=False)
print(f"Generated or found {len(structure_files)} CEMC structures.")

Generated or found 4 CEMC structures.


## Calculate Percolating Li Fraction


In [3]:
def write_dribble_input(json_file, structure_file):
    content = {
        "structure": str(Path(structure_file).resolve()),
        "formula_units": 1,
        "sublattices": {
            "cations": {
                "description": "Cation sites",
                "sites": {"species": ["Li"]},
                "initial_occupancy": {"Li": 1.0},
            },
            "cation2": {
                "description": "Transition-metal cation sites",
                "sites": {"species": ["Mn", "Ti"]},
                "initial_occupancy": {"TM": 1.0},
            },
            "oxygen": {
                "description": "Oxygen sites",
                "sites": {"species": ["O"]},
                "ignore": True,
            },
        },
        "bonds": [
            {
                "sublattices": ["cations", "cations"],
                "bond_rules": [["MinCommonNNNeighborsBR", {"num_neighbors": 2}]],
            }
        ],
        "percolating_species": ["Li"],
        "flip_sequence": [["TM", "Li"]],
    }
    Path(json_file).write_text(json.dumps(content, indent=4), encoding="utf-8")


def calculate_percolating_fraction(structure_file):
    input_json = Path(output_dir) / "input-bond-rule.json"
    write_dribble_input(input_json, structure_file)

    inp = Input.from_file(str(input_json))
    lattice = Lattice.from_input_object(inp, supercell=(1, 1, 1))
    percolator = Percolator.from_input_object(inp, lattice, verbose=False)
    n_li = percolator.num_occupied
    n_span = percolator.check_spanning(
        verbose=False,
        save_clusters=False,
        static_sites=inp.static_sites,
    )
    return n_span / n_li if n_li else np.nan


rows = []
for temperature, i, structure_file in structure_files:
    fraction = calculate_percolating_fraction(structure_file)
    rows.append({
        "temperature_K": temperature,
        "structure_index": i,
        "structure": str(structure_file),
        "percolating_li_fraction": fraction,
        "percolating_li_percent": 100 * fraction,
    })

results = pd.DataFrame(rows)
results

,temperature_K,structure_index,structure,percolating_li_fraction,percolating_li_percent
0,100,0,cemc_percolation_Li1p2Mn0p4Ti0p4O2/100K/CEMC_0...,0.0000,0.00
1,1000,0,cemc_percolation_Li1p2Mn0p4Ti0p4O2/1000K/CEMC_...,0.0000,0.00
2,10000,0,cemc_percolation_Li1p2Mn0p4Ti0p4O2/10000K/CEMC...,0.4375,43.75
3,100000,0,cemc_percolation_Li1p2Mn0p4Ti0p4O2/100000K/CEM...,0.8125,81.25


## Notes

The generated `CEMC_*.vasp` structures are written under `cemc_percolation_Li1p2Mn0p4Ti0p4O2/`. This folder is ignored by git because it is generated output. Increase `N_STRUCTURES` to 100 to reproduce the manuscript-scale statistics.
